## Dataset analysis: `student_attendance_list15.csv`

Purpose: daily attendance events (List15). Used to derive attendance trends per semester for CGPA prediction.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 200)
DATA_DIR = Path.cwd()
df = pd.read_csv(DATA_DIR / "student_attendance_list15.csv")
df.shape

(1078445, 5)

In [2]:
df.head()

,REG_NO,ACC_NO,DATE,SEMESTER_INDEX,STATUS
0,S24B33/010,B29465,2024-09-10,1,PRESENT
1,S24B33/010,B29465,2024-09-11,1,PRESENT
2,S24B33/010,B29465,2024-09-12,1,PRESENT
3,S24B33/010,B29465,2024-09-13,1,PRESENT
4,S24B33/010,B29465,2024-09-16,1,PRESENT


In [3]:
df["DATE"] = pd.to_datetime(df["DATE"], errors="coerce")
df["STATUS"] = df["STATUS"].astype(str).str.upper().str.strip().replace({"P":"PRESENT","A":"ABSENT","L":"LATE"})

df.isna().mean().sort_values(ascending=False)

REG_NO            0.0
ACC_NO            0.0
DATE              0.0
SEMESTER_INDEX    0.0
STATUS            0.0
dtype: float64

In [4]:
df["STATUS"].value_counts(dropna=False)

STATUS
PRESENT    1024492
LATE         32531
ABSENT       21422
Name: count, dtype: int64

In [5]:
# Semestral attendance rates
g = df.dropna(subset=["REG_NO","SEMESTER_INDEX","DATE"]).groupby(["REG_NO","SEMESTER_INDEX"], as_index=False)
out = g.agg(attendance_days=("DATE","nunique"), present=("STATUS", lambda s: (s=="PRESENT").sum()), late=("STATUS", lambda s: (s=="LATE").sum()), absent=("STATUS", lambda s: (s=="ABSENT").sum()))
out["present_rate"] = out["present"] / out["attendance_days"].replace({0: np.nan})
out[["attendance_days","present_rate"]].describe().T

,count,mean,std,min,25%,50%,75%,max
attendance_days,15691.0,68.730164,25.035472,5.0,79.000000,79.00,79.000000,80.0
present_rate,15691.0,0.950466,0.038063,0.6,0.936709,0.95,0.974684,1.0


## Advanced analytics

Focus: attendance trend features by semester and association with CGPA (joined to transcript).

In [ ]:
import numpy as np
from analysis_utils import basic_profile, missingness_report, merge_to_transcript_for_cgpa

print(basic_profile(df))
missingness_report(df, top_n=20)

BasicProfile(rows=1078445, cols=5, dup_rows=0, null_cells=0)


,dtype,missing_rate,missing_count,nunique
REG_NO,str,0.0,0,4978
ACC_NO,str,0.0,0,4978
DATE,datetime64[us],0.0,0,1500
SEMESTER_INDEX,int64,0.0,0,8
STATUS,str,0.0,0,3


In [ ]:
# Semester-level attendance rates joined to CGPA
trans = pd.read_csv(DATA_DIR / "student_transcript_list15.csv")
trans["CGPA"] = pd.to_numeric(trans["CGPA"], errors="coerce")

x = df.copy()
x["DATE"] = pd.to_datetime(x["DATE"], errors="coerce")
x["STATUS"] = x["STATUS"].astype(str).str.upper().str.strip().replace({"P":"PRESENT","A":"ABSENT","L":"LATE"})

x = x.dropna(subset=["REG_NO","SEMESTER_INDEX","DATE"])

g = x.groupby(["REG_NO","SEMESTER_INDEX"], as_index=False)
att_sem = g.agg(
    attendance_days=("DATE","nunique"),
    present_rate=("STATUS", lambda s: float((s=="PRESENT").mean())),
    late_rate=("STATUS", lambda s: float((s=="LATE").mean())),
    absent_rate=("STATUS", lambda s: float((s=="ABSENT").mean())),
    attendance_span_days=("DATE", lambda s: (s.max() - s.min()).days + 1),
)

joined = merge_to_transcript_for_cgpa(att_sem, trans, on=["REG_NO","SEMESTER_INDEX"], how="inner")
joined[["CGPA","present_rate","late_rate","absent_rate","attendance_days"]].corr(numeric_only=True)

,CGPA,present_rate,late_rate,absent_rate,attendance_days
CGPA,1.000000,0.009145,-0.008518,-0.003898,0.012895
present_rate,0.009145,1.000000,-0.755874,-0.633638,-0.035702
late_rate,-0.008518,-0.755874,1.000000,-0.027558,0.051290
absent_rate,-0.003898,-0.633638,-0.027558,1.000000,-0.006096
attendance_days,0.012895,-0.035702,0.051290,-0.006096,1.000000
